In [85]:
import osmnx as ox
import geopandas as gpd
import folium
import pandas as pd

# Gebiet
ort = "Basel, Switzerland"

# Gebäude laden
gebaeude = ox.features_from_place(
    ort,
    tags={"building": True}
)

gebaeude = gebaeude[
    gebaeude.geom_type.isin([
        "Polygon",
        "MultiPolygon"
    ])
]

print(gebaeude.columns)

Index(['geometry', 'addr:city', 'addr:housenumber', 'addr:postcode',
       'addr:street', 'building', 'building:levels', 'check_date', 'email',
       'fee',
       ...
       'type', 'name:es', 'diet:gluten_free', 'diet:vegan', 'diet:vegetarian',
       'drink:club-mate', 'payment:twint', 'name:ar', 'name:it',
       'heritage:operator'],
      dtype='str', length=233)


In [86]:
wichtige = gebaeude[
    (
        gebaeude["amenity"].isin([
            "hospital",
            "school",
            "fire_station",
            "police",
            "place_of_worship"   # ALLE religiösen Gebäude
        ])
    )
    |
    (
        gebaeude["building"].isin([
            "hospital",
            "church",
            "mosque",
            "synagogue",
            "temple",
            "chapel",
            "cathedral",
            "shrine"
        ])
    )
]

In [87]:
normale = gebaeude.drop(
    wichtige.index,
    errors="ignore"
)

In [88]:
karte_gebaude = folium.Map(
    location=[47.56, 7.59],
    zoom_start=13,
    tiles="OpenStreetMap"
)

In [89]:
folium.GeoJson(
    normale,
    style_function=lambda x: {
        "fillColor": "grey",
        "color": "grey",
        "weight": 0.3,
        "fillOpacity": 0.3
    },
    name="Gebäude"
).add_to(karte_gebaude)

In [90]:
folium.GeoJson(
    wichtige,
    style_function=lambda x: {
        "fillColor": "red",
        "color": "darkred",
        "weight": 1,
        "fillOpacity": 0.9
    },

    popup=folium.GeoJsonPopup(
        fields=[
            "name",
            "building",
            "amenity"
        ],

        aliases=[
            "Name",
            "Gebäude",
            "Typ"
        ]
    ),

    name="Wichtige Gebäude"

).add_to(karte_gebaude)

In [91]:
folium.LayerControl().add_to(karte_gebaude)

In [92]:
karte_gebaude.save("Gebaude.html")